# GeoMag ML — Google Colab T4 Runner

Thin wrapper over `ml_research/scripts/train.py` (no duplicated model or training code).

**Before running:**
1. Upload `ml_research_cloud_bundle.zip` (generated via `ml_research/scripts/pack_for_cloud.py`) to Google Drive: `/MyDrive/ONU_Thesis_ML/ml_research_cloud_bundle.zip`.
2. Select **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/ONU_Thesis_ML')
BUNDLE_ZIP = DRIVE_DIR / 'ml_research_cloud_bundle.zip'
WORK_DIR = Path('/content/ml_work')
OUT_DIR = Path('/content/artifacts_out')
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

!unzip -oq "{BUNDLE_ZIP}" -d "{WORK_DIR}"
%cd /content/ml_work
!pip install -q -e . --no-deps
!pip install -q -r requirements_colab_gpu.txt

In [ ]:
%cd /content/ml_work
!pytest -c pytest.ini tests -k "not bridge and not dataset_contract"

In [ ]:
%cd /content/ml_work
DATASET = 'data_cache/geomag_ml_oct2026_clean_v1.npz'

# Phase 3 Branch A: Perceiver-Bottleneck AE (4-fold base + key ablations)
!python scripts/train.py --config configs/phase3_branch_a_perceiver.yaml --dataset {DATASET} --fold all --ablation base,k3,k4,no_ut,beta_vae --device cuda --out-dir /content/artifacts_out

# Phase 3 Branch B: Cross-Attention Virtual Magnetometer (4-fold base + Zero-Shot + spatial bias ablation)
!python scripts/train.py --config configs/phase3_branch_b_cross_attn.yaml --dataset {DATASET} --fold all --ablation base,zero_shot,no_spatial_bias --device cuda --out-dir /content/artifacts_out

In [ ]:
import hashlib
from pathlib import Path

%cd /content/artifacts_out
ARCHIVE_PATH = Path('/content/drive/MyDrive/ONU_Thesis_ML/artifacts_p3_colab_t4.zip')
!zip -qr "{ARCHIVE_PATH}" .

h = hashlib.sha256()
with ARCHIVE_PATH.open('rb') as fh:
    for chunk in iter(lambda: fh.read(1024 * 1024), b''):
        h.update(chunk)
sha_file = ARCHIVE_PATH.with_suffix('.zip.sha256')
sha_file.write_text(f"{h.hexdigest()}  {ARCHIVE_PATH.name}\n", encoding='utf-8')
print('Saved:', ARCHIVE_PATH, f'({ARCHIVE_PATH.stat().st_size / (1024**2):.2f} MB)')
print('SHA256:', h.hexdigest())
print('After unzipping locally into ml_research/artifacts/, verify CPU parity via:')
print('  .venv_ml\\Scripts\\python.exe ml_research/scripts/evaluate.py --run-dir ml_research/artifacts/<run_id> --dataset ml_research/data_cache/geomag_ml_oct2026_clean_v1.npz --device cpu')